# 📊 Análisis de Evolución de Tickers por Categoría
Este notebook carga un CSV con snapshots intradiarios, rastrea la persistencia de cada ticker en sus categorías, calcula transiciones y detecta patrones repetibles.

**Requisitos:** `pandas`, `numpy`, `matplotlib`, `seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Cargar datos (ajusta el nombre si es necesario)
df = pd.read_csv('finviz_snapshots.csv')

# 2. Preprocesamiento
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['category'] = df['category'].str.strip()
df['ticker'] = df['ticker'].str.strip()
df = df.sort_values(['ticker', 'timestamp']).reset_index(drop=True)

print(f"📅 Rango: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"🔢 Snapshots: {df['timestamp'].nunique()} | 📈 Tickers: {df['ticker'].nunique()}")
df.head()

In [ ]:
# Categorías de interés
target_categories = [
    'Top Gainers', 'Top Losers', 'Most Active', 'Unusual Volume',
    'Overbought', 'Oversold', 'New High', 'New Low', 
    'Insider Buying', 'Insider Selling', 'Upgrades', 'Downgrades'
]
df = df[df['category'].isin(target_categories)].copy()

# Matriz de presencia: 1 si el ticker está en esa categoría en ese momento
presence = pd.crosstab(
    [df['ticker'], df['timestamp']], 
    df['category']
).fillna(0).astype(int).reset_index()

print("✅ Matriz de presencia creada. Dimensiones:", presence.shape)
presence.head()

In [ ]:
# Persistencia: ¿Cuántos snapshots mantiene un ticker en cada categoría?
persistence = df.groupby(['ticker', 'category'])['timestamp'].nunique().reset_index(name='snapshots')
total_snapshots = df['timestamp'].nunique()
persistence['persistence_pct'] = (persistence['snapshots'] / total_snapshots) * 100

high_persistence = persistence[persistence['persistence_pct'] >= 30].sort_values('persistence_pct', ascending=False)
print("🔍 Tickers con mayor persistencia (>=30% del día):")
high_persistence.head(15)

In [ ]:
# Matriz de transiciones entre categorías (consecutivas)
transitions = []
for ticker, grp in df.groupby('ticker'):
    grp = grp.sort_values('timestamp')
    timestamps = grp['timestamp'].unique()
    cat_sequence = [set(grp[grp['timestamp'] == t]['category']) for t in timestamps]
    
    for i in range(len(cat_sequence) - 1):
        for prev in cat_sequence[i]:
            for nxt in cat_sequence[i+1]:
                if prev != nxt:
                    transitions.append({'ticker': ticker, 'from': prev, 'to': nxt})

trans_df = pd.DataFrame(transitions)
trans_counts = trans_df.groupby(['from', 'to']).size().reset_index(name='count')
trans_matrix = trans_counts.pivot_table(index='from', columns='to', values='count', fill_value=0)
trans_prob = trans_matrix.div(trans_matrix.sum(axis=1), axis=0)

plt.figure(figsize=(10, 8))
sns.heatmap(trans_prob, annot=True, cmap="YlOrRd", fmt=".2f", linewidths=.5)
plt.title("🔁 Probabilidad de transición entre categorías")
plt.xlabel("Categoría siguiente")
plt.ylabel("Categoría actual")
plt.tight_layout()
plt.show()

In [ ]:
# Detección de patrones predefinidos
patterns = [
    (['Oversold', 'Unusual Volume'], ['Top Gainers', 'Most Active']),
    (['Insider Buying', 'Upgrades'], ['New High', 'Top Gainers']),
    (['Overbought', 'New High'], ['Top Losers', 'New Low']),
    (['Earnings Before', 'Unusual Volume'], ['Top Gainers', 'Top Losers'])
]

pattern_results = []
for early_cats, late_cats in patterns:
    matched_tickers = []
    for ticker, grp in df.groupby('ticker'):
        grp = grp.sort_values('timestamp')
        timestamps = grp['timestamp'].unique()
        mid = len(timestamps)//2
        
        has_early = any(set(grp[grp['timestamp'] == t]['category']).intersection(early_cats) 
                        for t in timestamps[:mid])
        has_late = any(set(grp[grp['timestamp'] == t]['category']).intersection(late_cats) 
                       for t in timestamps[mid:])
        
        if has_early and has_late:
            matched_tickers.append(ticker)
    
    pattern_results.append({
        'Patrón': f"{' | '.join(early_cats)} → {' | '.join(late_cats)}",
        'Tickers': len(matched_tickers),
        'Lista': ', '.join(matched_tickers[:10]) + ('...' if len(matched_tickers)>10 else '')
    })

pd.DataFrame(pattern_results)

In [ ]:
# Exportación
persistence.to_csv('resultados_persistencia.csv', index=False)
trans_counts.to_csv('resultados_transiciones.csv', index=False)
pd.DataFrame(pattern_results).to_csv('resultados_patrones.csv', index=False)
print("✅ Archivos exportados: resultados_persistencia.csv, resultados_transiciones.csv, resultados_patrones.csv")

## 📌 Cómo usar los resultados
- **`persistence_pct` > 40%**: Ticker con presencia constante. Ideal para momentum o mean-reversion.
- **Heatmap de transiciones**: Si `Oversold → Top Gainers` > 0.30, es una señal estadísticamente relevante.
- **Patrones**: Los tickers listados cumplieron la secuencia en el día analizado. Cruza con datos de cierre para validar win-rate.

💡 *Tip:* Filtra por `volume > 100000` en la celda 2 para evitar ruido de microcaps.